# Notebook 04 — Full Gradio Web Dashboard
### Phishing Email Forensic Analyzer & Link Defanger (Part 4 of 4)

This notebook ties everything together into an interactive **Gradio**
dashboard:

- **Input:** upload a `.eml` file *or* paste raw email source text.
- **Output:**
  1. A color-coded **Risk Badge** (Green / Orange / Red).
  2. A **Threat Summary** in Markdown listing every detected issue.
  3. A **DataFrame** of every extracted link, its defanged form, and
     whether its domain mismatches the sender's.

Run every cell in order. The last cell launches a public, shareable
Gradio URL (`share=True`) so you can demo this project from your
portfolio.

>  **Safety note:** Because links are automatically **defanged**
> before display, nothing in this dashboard's output is clickable —
> that's intentional and is standard practice for phishing triage
> tools.

## 1. Mount Drive, Install Dependencies, Import Modules

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q gradio tldextract pandas

import os, sys
PROJECT_ROOT = "/content/drive/MyDrive/Phishing_Analyzer_Project"
MODULES_DIR = os.path.join(PROJECT_ROOT, "modules")
sys.path.append(MODULES_DIR)

import importlib
import email_parser, link_defanger, nlp_analyzer, risk_scorer
importlib.reload(email_parser)
importlib.reload(link_defanger)
importlib.reload(nlp_analyzer)
importlib.reload(risk_scorer)

import gradio as gr
import pandas as pd

print("Gradio version:", gr.__version__)
print("All modules imported successfully.")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 1.3 MB/s eta 0:00:00
Gradio version: 6.26.0
All modules imported successfully.


## 2. Helper: Risk Badge HTML

In [2]:
def risk_badge_html(score, category, disabled_message=None):
    """
    Render a simple color-coded HTML badge:
      SAFE       -> green
      SUSPICIOUS -> orange
      DANGEROUS  -> red
    """
    colors = {"SAFE": "#2ecc71", "SUSPICIOUS": "#f39c12", "DANGEROUS": "#e74c3c"}
    color = colors.get(category, "#95a5a6")

    if disabled_message:
        return f"""
        <div style="padding:16px;border-radius:10px;background:#ecf0f1;
                    color:#2c3e50;font-family:sans-serif;border:1px solid #bdc3c7;">
          <strong> Unable to score:</strong> {disabled_message}
        </div>
        """

    return f"""
    <div style="padding:20px;border-radius:12px;background:{color};color:white;
                text-align:center;font-family:sans-serif;box-shadow:0 2px 6px rgba(0,0,0,0.15);">
      <div style="font-size:13px;letter-spacing:2px;opacity:0.9;">RISK LEVEL</div>
      <div style="font-size:34px;font-weight:bold;margin:4px 0;">{category}</div>
      <div style="font-size:18px;">{score} / 100</div>
    </div>
    """

EMPTY_LINKS_DF = pd.DataFrame(
    columns=["Original URL", "Defanged URL", "Link Domain", "Sender Domain", "Mismatch?"]
)
print("Badge helper ready.")

Badge helper ready.


## 3. Core Analysis Function

In [3]:
def analyze_email_source(file_obj, raw_text):
    """
    Main orchestration function wired to the Gradio "Analyze" button.
    Accepts an uploaded .eml file (Gradio gr.File value) and/or raw
    pasted email text, runs the full pipeline, and returns the three
    dashboard outputs: (badge_html, summary_markdown, links_dataframe).
    """
    # --- Resolve which input source to use ---
    raw_source = None
    error = None

    if file_obj is not None:
        try:
            file_path = file_obj.name if hasattr(file_obj, "name") else file_obj
            with open(file_path, "rb") as f:
                raw_source = f.read()
        except Exception as e:
            error = f"Could not read uploaded .eml file: {e}"
    elif raw_text and raw_text.strip():
        raw_source = raw_text
    else:
        error = "Please upload an .eml file or paste raw email source text before analyzing."

    if error:
        return risk_badge_html(0, "SAFE", disabled_message=error), f"**Error:** {error}", EMPTY_LINKS_DF

    # --- Parse ---
    parsed = email_parser.parse_email(raw_source)
    if parsed.get("error") and not parsed.get("body_text") and not parsed.get("body_html"):
        msg = parsed["error"]
        return risk_badge_html(0, "SAFE", disabled_message=msg), f"**Error:** {msg}", EMPTY_LINKS_DF

    full_text = email_parser.get_full_text_for_analysis(parsed)
    sender = parsed["headers"].get("from", "")

    # --- Analyze links, language, and compute the composite score ---
    link_results = link_defanger.analyze_links(full_text, sender)
    nlp_result = nlp_analyzer.analyze_text(full_text)
    risk = risk_scorer.compute_risk_score(nlp_result, link_results)

    # --- Build outputs ---
    badge_html = risk_badge_html(risk["score"], risk["category"])

    summary_lines = [
        f"### Threat Summary — **{risk['category']}** ({risk['score']}/100)",
        "",
        f"**From:** {parsed['headers'].get('from')}",
        "",
        f"**To:** {parsed['headers'].get('to')}",
        "",
        f"**Subject:** {parsed['headers'].get('subject')}",
        "",
        f"**Date:** {parsed['headers'].get('date')}",
        "",
        "**Findings:**",
    ]
    for reason in risk["reasons"]:
        summary_lines.append(f"- {reason}")

    if parsed.get("attachments"):
        summary_lines.append("")
        summary_lines.append(f"**Attachments detected:** {', '.join(parsed['attachments'])}")

    summary_markdown = "\n".join(summary_lines)

    if link_results:
        links_df = pd.DataFrame([
            {
                "Original URL": l["original_url"],
                "Defanged URL": l["defanged_url"],
                "Link Domain": l["link_domain"],
                "Sender Domain": l["sender_domain"],
                "Mismatch?": "YES" if l["domain_mismatch"] else "No",
            }
            for l in link_results
        ])
    else:
        links_df = EMPTY_LINKS_DF

    return badge_html, summary_markdown, links_df

print("Core analysis function ready.")

Core analysis function ready.


## 4. Build & Launch the Gradio Dashboard

In [ ]:
with gr.Blocks(title="Phishing Email Forensic Analyzer & Link Defanger") as app:
    gr.Markdown(
        """
        # Phishing Email Forensic Analyzer & Link Defanger
        Upload a raw `.eml` file **or** paste raw email source code below,
        then click **Analyze Email**. All links are automatically
        defanged in the output for safe review.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Input")
            file_input = gr.File(
                label="Upload .eml file",
                file_types=[".eml"],
                type="filepath",
            )
            text_input = gr.Textbox(
                label="...or paste raw email source",
                lines=15,
                placeholder="From: ...\nTo: ...\nSubject: ...\n\n<email body>",
            )
            analyze_btn = gr.Button("🔍 Analyze Email", variant="primary")

        with gr.Column(scale=1):
            gr.Markdown("### Results")
            badge_output = gr.HTML(label="Risk Badge")
            summary_output = gr.Markdown(label="Threat Summary")
            links_output = gr.Dataframe(
                label="Extracted Links (defanged)",
                headers=["Original URL", "Defanged URL", "Link Domain", "Sender Domain", "Mismatch?"],
                wrap=True,
            )

    analyze_btn.click(
        fn=analyze_email_source,
        inputs=[file_input, text_input],
        outputs=[badge_output, summary_output, links_output],
    )

app.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://70531855c01bcf8aad.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
